### imports and pandas settings

In [2]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [4]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [ ]:
batch_data = CF_OUTPUTS / "base_vs_thresholds" / "base_rf_lowthres_2026-05-05" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [ ]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [ ]:
# Define columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]

# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [7]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.42,,,,0.0633,
1,0,cf_1,,,,,,,18.7257,,,0.8884,1,False,0.0633,0.06
2,0,cf_2,,,,7,,,18.9745,,,0.8756,2,False,0.0633,0.07
3,0,cf_3,,,,7,,,18.1477,,,0.918,2,False,0.0633,0.05
4,0,cf_4,,,6,,,,17.4944,,,0.9515,2,False,0.0633,0.04
5,0,cf_5,,,,,,,17.1748,5,,0.9678,2,False,0.0633,0.0867
6,0,cf_6,3,,,,,,16.5468,,,1.0,2,False,0.0633,0.0967
7,0,cf_7,,,6,7,,,18.9745,,,0.8756,3,False,0.0633,0.0933
8,0,cf_8,,,,7,,,18.9745,7,,0.8756,3,True,0.0633,0.03
9,1,xin,3,4,1,2,3,0,22.3757,0,2.64,,,,0.1467,


### Filtering Valid CFs

In [8]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.42,,,,0.0633,
9,0,cf_8,,,,7,,,18.9745,7,,0.8756,3,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,2.64,,,,0.1467,
10,1,cf_3,,,,3,1,,22.3757,,,0.8796,3,True,0.1467,0.0
11,1,cf_5,,,2,,1,,22.3757,,,0.8796,3,True,0.1467,0.0233
12,1,cf_6,2,,,,1,,22.3757,,,0.8796,3,True,0.1467,0.01
13,1,cf_7,,,,,1,,22.3757,1,,0.8796,3,True,0.1467,0.05
2,2,xin,5,3,1,1,4,0,22.694,7,2.79,,,,0.1567,
14,2,cf_1,3,,,,2,,22.68,,,0.9712,3,True,0.1567,0.0133
15,2,cf_2,,,,6,1,,22.6757,,,1.0,3,True,0.1567,0.0633


### select one CF

In [9]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.42,,,,0.0633,
9,0,cf_8,,,,7,,,18.9745,7,,0.8756,3,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,2.64,,,,0.1467,
10,1,cf_5,,,2,,1,,22.3757,,,0.8796,3,True,0.1467,0.0233
2,2,xin,5,3,1,1,4,0,22.694,7,2.79,,,,0.1567,
11,2,cf_2,,,,6,1,,22.6757,,,1.0,3,True,0.1567,0.0633
3,3,xin,3,3,6,6,2,0,24.3418,1,2.86,,,,0.02,
12,3,cf_1,,,,,,,24.3375,,,0.8757,1,True,0.02,0.01
4,4,xin,5,4,2,7,1,0,21.2585,3,3.12,,,,0.0267,
13,4,cf_5,,3,,,,,21.2585,,,0.9868,2,True,0.0267,0.0133


In [10]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0

Original predicted risk: 0.0633

=== Counterfactuals ===

--- cf_8 ---
Predicted risk: 0.03
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0

Original predicted risk: 0.1467

=== Counterfactuals ===

--- cf_3 ---
Predicted risk: 0.0
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_5 ---
Predicted risk: 0.0233
Valid: True
Changes:
  cgtsmok: 1 → 2
  slprl: 3 → 1

--- cf_6 ---
Predicted risk: 0.01
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_7 ---
Predicted risk: 0.05
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1


=== Query index '2' ===
T